# OLS and GLMs with statsmodels

## Overview

`statsmodels` provides R-style formula-based model fitting with full inference output — p-values, confidence intervals, and diagnostics — making it the natural Python choice for confirmatory statistical modelling.

**Model families covered:**

| Model | Response | Link | Use when |
|---|---|---|---|
| OLS | Continuous | Identity | Normal errors, homoscedastic |
| Logistic | Binary 0/1 | Logit | Binary outcome |
| Poisson | Count ≥ 0 | Log | Count data, no overdispersion |
| Negative Binomial | Count ≥ 0 | Log | Count data with overdispersion |
| Gamma | Continuous > 0 | Log | Right-skewed positive continuous |

**Formula interface:** `"y ~ x1 + x2 + x1:x2"` mirrors R's `lm()`/`glm()` syntax exactly.

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

rng = np.random.default_rng(42)
n = 200
elevation   = rng.uniform(50, 400, n)
nitrate     = rng.gamma(2, 2, n)
treatment   = rng.choice([0, 1], n)
catchment   = rng.choice(['North','South','East','West'], n)
richness    = (25 - 0.04*elevation - 0.8*nitrate + 3.0*treatment
               + rng.normal(0, 3.5, n)).clip(0)
restored    = (richness > richness.mean()).astype(int)
invertcount = rng.negative_binomial(
    n=5, p=0.4, size=n) + (treatment * rng.poisson(3, n))
df = pd.DataFrame(dict(elevation=elevation, nitrate=nitrate,
    treatment=treatment, catchment=catchment,
    richness=richness, restored=restored, invertcount=invertcount))
print(df.describe().round(2))

       elevation  nitrate  treatment  richness  restored  invertcount
count     200.00   200.00     200.00    200.00    200.00       200.00
mean      222.85     4.16       0.52     14.11      0.52         9.09
std        98.81     2.93       0.50      6.13      0.50         4.94
min        52.58     0.19       0.00      0.00      0.00         1.00
25%       138.28     2.16       0.00     10.02      0.00         5.00
50%       217.65     3.58       1.00     14.67      1.00         8.00
75%       311.54     5.52       1.00     18.06      1.00        12.00
max       397.33    17.87       1.00     31.40      1.00        27.00


---
## OLS: Linear Regression

In [2]:
# Formula interface -- same syntax as R's lm()
ols = smf.ols("richness ~ elevation + nitrate + C(treatment) + C(catchment)",
              data=df).fit()
print(ols.summary())
# Confidence intervals
print("\n95% Confidence intervals:")
print(ols.conf_int().round(3))

                            OLS Regression Results                            
Dep. Variable:               richness   R-squared:                       0.707
Model:                            OLS   Adj. R-squared:                  0.698
Method:                 Least Squares   F-statistic:                     77.72
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           7.92e-49
Time:                        08:43:42   Log-Likelihood:                -523.17
No. Observations:                 200   AIC:                             1060.
Df Residuals:                     193   BIC:                             1083.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                24.35

---
## Logistic and Poisson GLMs

In [3]:
# Logistic regression: binary outcome
logit = smf.logit("restored ~ elevation + nitrate + C(treatment)",
                   data=df).fit()
print(logit.summary().tables[1])
# Odds ratios
OR = np.exp(logit.params)
OR_ci = np.exp(logit.conf_int())
print("\nOdds Ratios:")
print(pd.concat([OR.rename('OR'), OR_ci], axis=1).round(3))

# Poisson GLM for counts
pois = smf.glm("invertcount ~ elevation + nitrate + C(treatment)",
               data=df,
               family=sm.families.Poisson()).fit()
print("\nPoisson deviance:", round(pois.deviance, 2))
print("Pearson chi2:    ", round(pois.pearson_chi2, 2))
print("Dispersion ratio:", round(pois.pearson_chi2 / pois.df_resid, 3),
      " (>1.5 suggests overdispersion -> use NegBin)")

Optimization terminated successfully.
         Current function value: 0.280613
         Iterations 8
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             7.6710      1.224      6.267      0.000       5.272      10.070
C(treatment)[T.1]     2.5620      0.566      4.526      0.000       1.452       3.672
elevation            -0.0290      0.004     -6.609      0.000      -0.038      -0.020
nitrate              -0.5512      0.122     -4.519      0.000      -0.790      -0.312

Odds Ratios:
                         OR        0          1
Intercept          2145.228  194.764  23628.561
C(treatment)[T.1]    12.962    4.274     39.315
elevation             0.971    0.963      0.980
nitrate               0.576    0.454      0.732

Poisson deviance: 468.64
Pearson chi2:     478.67
Dispersion ratio: 2.442  (>1.5 suggests overdispersion -> use NegBin)


---
## Negative Binomial for Overdispersed Counts

In [4]:
# Negative binomial handles overdispersion
nb = smf.glm("invertcount ~ elevation + nitrate + C(treatment)",
             data=df,
             family=sm.families.NegativeBinomial()).fit()
print("Negative Binomial summary:")
print(nb.summary().tables[1])

# Compare AIC
print(f"\nAIC comparison:")
print(f"  Poisson:           {pois.aic:.2f}")
print(f"  Negative Binomial: {nb.aic:.2f}")
print("Lower AIC = better fit; NegBin should win with overdispersed counts")

# Incidence rate ratios (exponentiated NegBin coefficients)
IRR = np.exp(nb.params)
IRR_ci = np.exp(nb.conf_int())
print("\nIncidence Rate Ratios:")
print(pd.concat([IRR.rename('IRR'), IRR_ci], axis=1).round(3))

Negative Binomial summary:
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             2.0488      0.228      8.994      0.000       1.602       2.495
C(treatment)[T.1]     0.3490      0.150      2.325      0.020       0.055       0.643
elevation          9.752e-07      0.001      0.001      0.999      -0.001       0.001
nitrate              -0.0090      0.026     -0.350      0.727      -0.059       0.041

AIC comparison:
  Poisson:           1257.32
  Negative Binomial: 1306.44
Lower AIC = better fit; NegBin should win with overdispersed counts

Incidence Rate Ratios:
                     IRR      0       1
Intercept          7.759  4.965  12.125
C(treatment)[T.1]  1.418  1.056   1.903
elevation          1.000  0.999   1.001
nitrate            0.991  0.942   1.042


c:\Users\saman\Documents\Repos\python_methods_library\.venv\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [5]:
# Gamma GLM: continuous positive response (e.g. nitrate concentration)
# Simulate a right-skewed positive response
biomass = rng.gamma(shape=2 + 0.005*elevation, scale=1.5, size=n)
df['biomass'] = biomass
gamma_mod = smf.glm("biomass ~ elevation + C(treatment)",
                    data=df,
                    family=sm.families.Gamma(link=sm.families.links.Log())).fit()
print("Gamma GLM (log link):")
print(gamma_mod.summary().tables[1])
print(f"\nDispersion parameter: {gamma_mod.scale:.4f}")
print("Dispersion > 1 with Gamma is expected and estimated, not a problem")

Gamma GLM (log link):
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             1.3927      0.105     13.246      0.000       1.187       1.599
C(treatment)[T.1]    -0.1377      0.076     -1.808      0.071      -0.287       0.012
elevation             0.0011      0.000      2.755      0.006       0.000       0.002

Dispersion parameter: 0.2870
Dispersion > 1 with Gamma is expected and estimated, not a problem


---

## Common Pitfalls

**1. Using OLS for binary or count outcomes**  
OLS applied to a 0/1 outcome can produce predicted probabilities outside [0,1] and violates homoscedasticity. OLS on count data underestimates variance at high counts. Always match the GLM family to the distributional properties of the response.

**2. Ignoring overdispersion in Poisson models**  
Poisson assumes mean = variance. When the variance is substantially larger (dispersion ratio > 1.5), Poisson standard errors are too small and p-values too optimistic. Always check the Pearson chi-squared / residual df ratio; use Negative Binomial or quasi-Poisson when overdispersed.

**3. Interpreting GLM coefficients on the link scale as effect sizes**  
Logistic regression coefficients are log-odds; Poisson/NegBin coefficients are log rate ratios. Always exponentiate to get odds ratios or incidence rate ratios before interpreting effect sizes, and report both the coefficient and the exponentiated form.

**4. Not checking residual deviance against degrees of freedom**  
For a well-fitting GLM, residual deviance / df should be approximately 1. A ratio >> 1 indicates underfitting or overdispersion; a ratio << 1 (rare) indicates overfitting. Always inspect this ratio in the summary before trusting inference.

**5. Using `C()` in formulas without verifying the reference level**  
`C(treatment)` uses the first alphanumeric level as the reference. If "control" sorts after "restored", the reference level is "restored" — the opposite of the intended contrast. Always check `model.model.data.param_names` or set the reference explicitly with `C(treatment, Treatment("control"))`.
---
*python_methods_library - Samantha McGarrigle*